# 유망아이템 발굴 실습 — 0. 소개 & API 준비

KISTI **APOLLO** 공식 Open API로 *유망아이템 발굴* 전 과정을 단계별로 배웁니다.
기존 위키피디아 크롤링 방식을 공식 API 호출로 대체해, 빠르고 재현 가능하게 실습합니다.

## 전체 6단계
| 단계 | 내용 | 사용 API |
|---|---|---|
| 1 | 시드 아이템 확정 | A4.1 TOP100 |
| 2 | 연관 네트워크 확장 | A4.4 network |
| 3 | 아이템 상세 수집 | A4.5 network/list |
| 4 | 네트워크 그래프 시각화 | (2·3단계 결과) |
| 5 | 종합 스코어링 | (지표 종합) |
| 6 | AI 보고서 생성 | 로컬 LLM(Qwen) |

## APOLLO A4 (글로벌 유망 아이템 탐색)
- Base URL: `https://apollo.kisti.re.kr/service-test`
- 지표 3종: **기술집약도(techIntensity)** · **수요부상도(demandEmergence)** · **공급부상도(supplyEmergence)**
- 응답 형식: `{"meta": {...}, "data": ...}`
- ⚠️ 테스트 서버는 사설 인증서라 `verify=False`로 호출합니다(KISTI 망 전제).

In [ ]:
# --- APOLLO API 호출 헬퍼 (모든 노트북 공통) ---
import requests, urllib3
import pandas as pd
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

BASE_URL = "https://apollo.kisti.re.kr/service-test"   # 테스트 서버

def call_api(method, path, params=None, payload=None):
    """APOLLO 호출 후 {meta, data} 본문을 반환. (테스트 서버라 verify=False)"""
    r = requests.request(method, BASE_URL + path, params=params, json=payload,
                         headers={"Content-Type": "application/json"},
                         verify=False, timeout=120)
    body = r.json()
    if not (isinstance(body, dict) and "data" in body):
        raise RuntimeError(f"API 오류 (HTTP {r.status_code}): {body}")
    return body

print("준비 완료 · BASE_URL =", BASE_URL)

## 도달 확인 — TOP100 한 건 불러오기

In [ ]:
# 인공지능 카테고리 기술집약도 TOP100 (첫 1건만 확인)
res = call_api("GET", "/open/api/v1/itemsntop100",
               params={"category": "인공지능", "indicator": "TECH_INTENSITY"})
items = res["data"]["items"]
print("TOP100 수신:", len(items), "건")
pd.DataFrame(items).head(3)[["rank", "itemId", "itemName", "score"]]

정상적으로 데이터가 오면 준비 완료입니다. 이제 **1단계**부터 순서대로 진행하세요.

각 노트북은 독립 실행되도록 `call_api` 헬퍼를 상단에 포함합니다.